In [1]:
# =================================================================
# 03b_chronicle_rain_analysis_and_standardization
# Actions: 
# 1. Standardize area (Mollweide) and recalculate PFDI for consistency.
# 2. Perform Spatial-Temporal Rainfall Intensity Analysis (30m - 24h).
# Note: Includes a 'Clean Slate' step to remove polluted intensity columns.
# =================================================================

import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# --- 1. CONFIGURATION ---
RAIN_INPUT_PATH = r"D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs\chronicle_urban_df_with_IMERG_FULL.pkl"
RAIN_MASTER_FILE_PATH = r"D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs\chronicle_rain_master.pkl"

# Durations in minutes for peak intensity calculation
DURATIONS = [30, 60, 120, 240, 360, 720, 1440]

# --- 2. LOAD DATA ---
print(f"Loading chronicle rainfall data: {RAIN_INPUT_PATH}")
chronicle_events = pd.read_pickle(RAIN_INPUT_PATH)

Loading chronicle rainfall data: D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs\chronicle_urban_df_with_IMERG_FULL.pkl


In [2]:
chronicle_events.columns

Index(['Unnamed: 0', 'uuid', 'area_km2', 'version', 'start_time', 'end_time',
       'duration_days', 'geometry_wkt', 'urban_built_up_area_m2',
       'polygon_total_area_m2', 'urban_percentage', 'event_id',
       'poly_area_km2', 'upa_max', 'upa_p95', 'upa_p99', 'PFDI_p95',
       'PFDI_p99', 'PFDI_max', 'imerg_matrix', 'imerg_mask', 'imerg_meta'],
      dtype='object')

In [3]:
# =================================================================
# STEP 0: CLEAN SLATE (PREVENT COLUMN POLLUTION)
# =================================================================
# Removing any old intensity columns to ensure we don't carry over logic errors
old_intensity_cols = [c for c in chronicle_events.columns if 'max_rainfall_intens' in c]
if old_intensity_cols:
    print(f"Removing {len(old_intensity_cols)} existing intensity columns for a clean run...")
    chronicle_events = chronicle_events.drop(columns=old_intensity_cols)

# =================================================================
# ACTION 1: AREA STANDARDIZATION & MULTI-PFDI SYNC
# =================================================================
print("Step 1: Standardizing area and syncing PFDI metrics (p95, p99, max)...")

# Update master area to km2 using the precision Mollweide area
chronicle_events['area_km2'] = chronicle_events['polygon_total_area_m2'] / 1e6

# Synchronizing all PFDI metrics with the new standardized area
metrics_to_sync = [
    ('upa_p95', 'PFDI_p95'),
    ('upa_p99', 'PFDI_p99'),
    ('upa_max', 'PFDI_max')
]

for upa_col, pfdi_col in metrics_to_sync:
    if upa_col in chronicle_events.columns:
        chronicle_events[pfdi_col] = np.where(
            chronicle_events['area_km2'] > 0,
            chronicle_events[upa_col] / chronicle_events['area_km2'],
            0
        )

# Remove redundant columns to keep the dataset lean
redundant_cols = ['polygon_total_area_m2', 'poly_area_km2', 'Unnamed: 0', 'version']
chronicle_events = chronicle_events.drop(columns=[c for c in redundant_cols if c in chronicle_events.columns])

# =================================================================
# ACTION 2: INTENSITY ANALYSIS (Spatial then Temporal)
# =================================================================
print(f"Step 2: Calculating Peak Intensities for {len(chronicle_events)} events...")

event_intensity_list = []

for idx, row in tqdm(chronicle_events.iterrows(), total=len(chronicle_events)):
    rain_matrix = row['imerg_matrix']
    polygon_mask = row['imerg_mask']
    
    if not isinstance(rain_matrix, np.ndarray) or rain_matrix.size == 0 or polygon_mask.sum() == 0:
        continue

    # Spatial average across the entire polygon mask
    spatial_mean_series = np.nanmean(rain_matrix[:, polygon_mask == 1], axis=1)
    hyetograph = pd.Series(spatial_mean_series)

    peak_stats = {'event_id': row['event_id']}
    
    # Temporal rolling max for each duration
    for duration_min in DURATIONS:
        window_steps = int(duration_min / 30)
        if len(hyetograph) >= window_steps:
            peak_val = hyetograph.rolling(window=window_steps).mean().max()
            peak_stats[f"{duration_min}_max_rainfall_intens"] = peak_val
        else:
            peak_stats[f"{duration_min}_max_rainfall_intens"] = np.nan
            
    event_intensity_list.append(peak_stats)

# =================================================================
# FINAL MERGE & SAVE
# =================================================================
intensity_summary = pd.DataFrame(event_intensity_list)
final_master_dataset = chronicle_events.merge(intensity_summary, on='event_id', how='left')

print(f"\nProcessing Complete. Total events: {len(final_master_dataset)}")
final_master_dataset.to_pickle(RAIN_MASTER_FILE_PATH)
print(f"SUCCESS! Clean master dataset saved to: {RAIN_MASTER_FILE_PATH}")

Step 1: Standardizing area and syncing PFDI metrics (p95, p99, max)...
Step 2: Calculating Peak Intensities for 700912 events...


100%|█████████████████████████████████████████████████████████████████████████| 700912/700912 [15:01<00:00, 777.87it/s]



Processing Complete. Total events: 700912
SUCCESS! Clean master dataset saved to: D:\Development\RESEARCH\urban_flood_database\chronicle\imerg_rain_outputs\chronicle_rain_master.pkl


In [4]:
final_master_dataset

,uuid,area_km2,start_time,end_time,duration_days,geometry_wkt,urban_built_up_area_m2,urban_percentage,event_id,upa_max,...,imerg_matrix,imerg_mask,imerg_meta,30_max_rainfall_intens,60_max_rainfall_intens,120_max_rainfall_intens,240_max_rainfall_intens,360_max_rainfall_intens,720_max_rainfall_intens,1440_max_rainfall_intens
0,1422c1c0986549d7a3c7d10a09208098,17.902320,2000-06-01,9.598176e+08,1,"POLYGON ((-98.923354 19.259108, -98.916374 19....",6055133.0,33.823174,296,7775.601562,...,"[[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]], [[0...","[[1, 0]]","{'origin_top_left': (19.288329, -98.923354), '...",4.568446,4.568446,3.679287,2.567239,1.828248,1.171332,0.742607
1,43553235080041299c3e48978885c635,29.849550,2000-06-01,9.598176e+08,1,"POLYGON ((-98.953929 19.329468, -98.883774 19....",8815066.0,29.531655,297,7266.895020,...,"[[[0.33306354, 0.0024441965], [0.0, 0.0]], [[0...","[[1, 0], [0, 0]]","{'origin_top_left': (19.346044, -98.953929), '...",5.905882,5.328168,4.617779,3.525245,2.631945,1.408315,0.844070
2,6532dc29aaa645239f52ad164489e0aa,1760.002208,2000-06-01,9.598176e+08,1,"POLYGON ((95.067367 28.023449, 95.288819 28.09...",6223800.0,0.353625,298,255259.687500,...,"[[[0.0, 0.39326465, 1.6282946, 0.2216488, 0.23...","[[0, 1, 1, 1, 1, 0], [0, 1, 1, 1, 1, 0], [1, 1...","{'origin_top_left': (28.320151, 95.067367), 's...",1.239642,0.898038,0.698607,0.636494,0.466378,0.268521,0.134261
3,bd7267cb286448798c31cfab0f6a0f95,11.800292,2000-06-01,9.598176e+08,1,"POLYGON ((-98.97432499999999 19.337026, -98.93...",3757833.0,31.845253,299,8171.314453,...,"[[[0.33306354], [0.0]], [[0.5504179], [0.0]], ...","[[1], [0]]","{'origin_top_left': (19.337026, -98.974325), '...",5.905882,5.328168,4.617779,3.525245,2.631945,1.408315,0.844070
4,d2205ca5dd1c42f2b6ca67ab9bf1e9e8,105.652324,2000-06-01,9.598176e+08,1,"POLYGON ((1.3503293 43.604335, 1.4399303 43.66...",28337179.0,26.821160,300,10537.763672,...,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], [[0.0, 0....","[[1, 1, 0], [1, 1, 0]]","{'origin_top_left': (43.668714, 1.3503293), 's...",20.037289,15.327168,9.840519,4.920259,3.280173,1.640086,0.820043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
700907,6844af49acb046a192c84af186ffac65,0.447648,2024-06-05,1.717546e+09,1,"POLYGON ((-73.19565799999999 -37.285529, -73.2...",14946.0,3.338781,702775,34.354675,...,"[[[0.0, 0.0]], [[0.0, 0.0]], [[0.0, 0.0]], [[0...","[[1, 0]]","{'origin_top_left': (-37.273155, -73.200259), ...",0.594314,0.297157,0.200247,0.121922,0.082700,0.041350,NaN
700908,67f5fa5b27db40bca00cff4084f80e2c,230.548112,2024-06-05,1.717546e+09,1,"POLYGON ((76.96611 10.937957, 76.8721100000000...",56655765.0,24.574378,702776,460.582947,...,"[[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0,...","[[1, 1, 1], [1, 1, 0], [0, 0, 0]]","{'origin_top_left': (11.114692, 76.87211), 'sc...",12.016616,11.845717,9.705459,5.277455,3.836622,2.053450,NaN
700909,696b5ca527c04590a0cfe9716cc299c5,12.076412,2024-06-05,1.717546e+09,1,"POLYGON ((104.82588 21.831464, 104.80142 21.78...",63623.0,0.526837,702777,44595.226562,...,"[[[0.0, 0.0], [0.0, 0.0]], [[0.0, 0.0], [0.0, ...","[[1, 0], [0, 0]]","{'origin_top_left': (21.831464, 104.77429), 's...",0.017614,0.008807,0.004403,0.002202,0.001468,0.000734,NaN
700910,68a415ff17ec49519d39a9e6a5918fb9,3.967753,2024-06-05,1.717546e+09,1,"POLYGON ((-38.480898 -12.859858, -38.456067 -1...",1066951.0,26.890563,702778,10.196815,...,"[[[0.0]], [[0.0]], [[0.0]], [[0.0]], [[0.0]], ...",[[1]],"{'origin_top_left': (-12.854418, -38.480898), ...",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
